# Classical ML Baselines
Sometimes simple models work surprisingly well!

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    f1_score,
    precision_recall_fscore_support
)
from sklearn.preprocessing import LabelEncoder
import joblib
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# For XGBoost (install if needed)
try:
    import xgboost as xgb
    print("✅ XGBoost available")
except ImportError:
    print("⚠️  XGBoost not installed. Run: pip install xgboost")

print("✅ All libraries loaded!")

## Load and Prepare Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/customer_support_clean.csv')

print(f" Dataset shape: {df.shape}")
print(f" Number of categories: {df['category'].nunique()}")

# Check for any nulls
print(f"\n Missing values:\n{df.isnull().sum()}")

# Preview
df.head()

## Prepare features and labels

In [ ]:
# Features: customer queries (instructions)
X = df['instruction'].values

# Labels: categories we want to predict
y = df['category'].values

print(f"Features (X): {X.shape}")
print(f"Labels (y): {y.shape}")
print(f"\n Sample query: {X[0]}")
print(f" Sample label: {y[0]}")

## Encode labels to numbers

In [ ]:
# ML models need numeric labels, not text
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Labels encoded to numbers: 0 to {y_encoded.max()}")
print(f"\nExample mapping:")
for i in range(min(5, len(label_encoder.classes_))):
    print(f"  {label_encoder.classes_[i]} → {i}")

# Save encoder for later use
joblib.dump(label_encoder, '../models/label_encoder.pkl')
print("\n Label encoder saved to: models/label_encoder.pkl")

## Train/Validation/Test Split

In [ ]:
# 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.15, 
    random_state=42,
    stratify=y_encoded  # Keep same class distribution
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 0.176 of 85% ≈ 15% of total
    random_state=42,
    stratify=y_temp
)

print("DATASET SPLITS")
print("=" * 60)
print(f"Training set:   {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set: {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set:       {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

# Verify class distribution is maintained
print(f"\nClass distribution check:")
print(f"Train: {Counter(y_train).most_common(3)}")
print(f"Val:   {Counter(y_val).most_common(3)}")
print(f"Test:  {Counter(y_test).most_common(3)}")

## Feature Engineering - TF-IDF

In [ ]:
print("Creating TF-IDF features...")

# TF-IDF: Term Frequency - Inverse Document Frequency
# Gives more weight to rare, informative words
tfidf = TfidfVectorizer(
    max_features=5000,      # Keep top 5000 words
    ngram_range=(1, 2),     # Use single words and pairs
    min_df=2,               # Word must appear in at least 2 documents
    max_df=0.95,            # Ignore words in >95% of documents
    stop_words='english'    # Remove common words like 'the', 'a'
)

# Fit on training data only!
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f" TF-IDF shape: {X_train_tfidf.shape}")
print(f"   Features: {X_train_tfidf.shape[1]:,} words/phrases")
print(f"   Sparsity: {(1 - X_train_tfidf.nnz / np.prod(X_train_tfidf.shape)) * 100:.1f}%")

# Save vectorizer
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
print("TF-IDF vectorizer saved")

## Inspect top features

In [ ]:
feature_names = tfidf.get_feature_names_out()
print(f"Sample features (words/phrases):")
print(feature_names[100:120])

# Show most common unigrams and bigrams
vocab = tfidf.vocabulary_
print(f"\n Vocabulary size: {len(vocab):,}")

## Experiment 1A - Logistic Regression (Baseline)

In [ ]:
print("EXPERIMENT 1A: Logistic Regression")
print("=" * 60)

start_time = time.time()

# Create and train model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',  # Handle class imbalance
    n_jobs=-1  # Use all CPU cores
)

lr_model.fit(X_train_tfidf, y_train)

train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")

# Predictions
y_val_pred = lr_model.predict(X_val_tfidf)

# Evaluate
accuracy = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average='macro')
f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f"\n VALIDATION RESULTS")
print(f"Accuracy:        {accuracy:.4f}")
print(f"F1 (Macro):      {f1_macro:.4f}")
print(f"F1 (Weighted):   {f1_weighted:.4f}")

# Save model
joblib.dump(lr_model, '../models/lr_tfidf.pkl')
print("\nModel saved to: models/lr_tfidf.pkl")

## Detailed classification report

In [ ]:
print(" DETAILED CLASSIFICATION REPORT")
print("=" * 60)

# Get top 10 most common classes for detailed view
top_classes = Counter(y_val).most_common(10)
top_class_indices = [idx for idx, _ in top_classes]

report = classification_report(
    y_val, 
    y_val_pred,
    target_names=label_encoder.classes_,
    digits=3
)
print(report)

## Confusion matrix for top categories

In [ ]:
# Visualize where model makes mistakes
top_10_classes = label_encoder.inverse_transform(top_class_indices)

# Filter predictions for top 10 classes only
mask = np.isin(y_val, top_class_indices)
y_val_top10 = y_val[mask]
y_pred_top10 = y_val_pred[mask]

# Create confusion matrix
cm = confusion_matrix(y_val_top10, y_pred_top10, labels=top_class_indices)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=top_10_classes,
    yticklabels=top_10_classes
)
plt.title('Confusion Matrix - Top 10 Categories\n(Logistic Regression)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../experiments/lr_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to: experiments/lr_confusion_matrix.png")

## Experiment 1B - Random Forest

In [ ]:
# Cell 11: Train Random Forest
print("EXPERIMENT 1B: Random Forest")
print("=" * 60)

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,        # Number of trees
    max_depth=30,            # Prevent overfitting
    min_samples_split=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)

train_time = time.time() - start_time

# Predictions
y_val_pred_rf = rf_model.predict(X_val_tfidf)

# Evaluate
accuracy_rf = accuracy_score(y_val, y_val_pred_rf)
f1_macro_rf = f1_score(y_val, y_val_pred_rf, average='macro')
f1_weighted_rf = f1_score(y_val, y_val_pred_rf, average='weighted')

print(f" Training completed in {train_time:.2f} seconds")
print(f"\n VALIDATION RESULTS")
print(f"Accuracy:        {accuracy_rf:.4f}")
print(f"F1 (Macro):      {f1_macro_rf:.4f}")
print(f"F1 (Weighted):   {f1_weighted_rf:.4f}")

# Save
joblib.dump(rf_model, '../models/rf_tfidf.pkl')

## Experiment 1C - XGBoost

In [ ]:
print("EXPERIMENT 1C: XGBoost")
print("=" * 60)

start_time = time.time()

# Calculate class weights for imbalanced data
class_counts = Counter(y_train)
total_samples = len(y_train)
n_classes = len(class_counts)
class_weights = {
    cls: total_samples / (n_classes * count) 
    for cls, count in class_counts.items()
}

# Create sample weights
sample_weights = np.array([class_weights[cls] for cls in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    tree_method='hist',  # Faster training
    n_jobs=-1
)

xgb_model.fit(
    X_train_tfidf, 
    y_train,
    sample_weight=sample_weights
)

train_time = time.time() - start_time

# Predictions
y_val_pred_xgb = xgb_model.predict(X_val_tfidf)

# Evaluate
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)
f1_macro_xgb = f1_score(y_val, y_val_pred_xgb, average='macro')
f1_weighted_xgb = f1_score(y_val, y_val_pred_xgb, average='weighted')

print(f"Training completed in {train_time:.2f} seconds")
print(f"\nVALIDATION RESULTS")
print(f"Accuracy:        {accuracy_xgb:.4f}")
print(f"F1 (Macro):      {f1_macro_xgb:.4f}")
print(f"F1 (Weighted):   {f1_weighted_xgb:.4f}")

# Save
joblib.dump(xgb_model, '../models/xgb_tfidf.pkl')

## Compare All Models

In [2]:
## Remember to calculate detailed classification report and confusion matrix for all 11 classes